<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_3/blob/main/17_6_3_THEORY_LangChain_Q%26A_Over_Documents_(LLama2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain > /dev/null
!pip install transformers > /dev/null
!pip install pdf2image > /dev/null
!pip install pdfminer.six > /dev/null
!pip install pillow > /dev/null
!pip install unstructured > /dev/null
!pip install sentence_transformers > /dev/null
!pip install faiss-gpu || pip install faiss-cpu
!pip install accelerate > /dev/null
!pip install bitsandbytes > /dev/null
!pip install langchain-community
!pip install pi-heif
!pip install unstructured-inference

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00


In [ ]:
import os
import sys
import textwrap

from huggingface_hub import notebook_login
from langchain import HuggingFacePipeline
from langchain.chains import RetrievalQA, RetrievalQAWithSourcesChain
from langchain.chains.question_answering import load_qa_chain
from langchain_community.document_loaders import UnstructuredFileLoader, PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

In [ ]:
from getpass import getpass

huggingface_api_token = getpass('Insert your HuggingFace API Token: ')

Insert your HuggingFace API Token: ··········


We are going to make questions about the latest NVIDIA financial quarterly report that you can download from the following [LINK](https://nvidianews.nvidia.com/_gallery/download_pdf/630688d4ed6ae52ca35fe13f/)

In [ ]:
!wget https://images.nvidia.com/nvimages/aem-dam/Solutions/about-us/documents/NVIDIA.pdf

--2025-07-11 00:27:02--  https://images.nvidia.com/nvimages/aem-dam/Solutions/about-us/documents/NVIDIA.pdf
Resolving images.nvidia.com (images.nvidia.com)... 23.59.88.2, 23.59.88.14
Connecting to images.nvidia.com (images.nvidia.com)|23.59.88.2|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17468362 (17M) [application/pdf]
Saving to: ‘NVIDIA.pdf.2’

NVIDIA.pdf.2        100%[===================>]  16.66M  96.2MB/s    in 0.2s    

2025-07-11 00:27:03 (96.2 MB/s) - ‘NVIDIA.pdf.2’ saved [17468362/17468362]



In [ ]:
loader = PyPDFLoader("NVIDIA.pdf")
documents = loader.load()  # returns one Document per page
print(f"Loaded {len(documents)} pages")
print(documents[0].page_content[:200])  # preview

Loaded 38 pages



In [ ]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
)
text_chunks=text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cuda'},
)


/tmp/ipython-input-10-2076742005.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = FAISS.from_documents(text_chunks, embeddings)

In [ ]:
import torch
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    token=huggingface_api_token,
)

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    device_map='auto',
    torch_dtype=torch.float16,
    load_in_8bit=True,
    token=huggingface_api_token,
)

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    max_new_tokens=1024,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

Device set to use cuda:0


In [ ]:
llm=HuggingFacePipeline(pipeline=pipe, model_kwargs={'temperature':0})

/tmp/ipython-input-19-596095089.py:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm=HuggingFacePipeline(pipeline=pipe, model_kwargs={'temperature':0})


In [ ]:
chain =  RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    return_source_documents=True,
    retriever=vectorstore.as_retriever()
)

In [ ]:
query = "When will NVIDIA pay the next quarterly dividend?"
result=chain({"query": query}, return_only_outputs=True)
wrapped_text = textwrap.fill(result['result'], width=500)
wrapped_text

/tmp/ipython-input-21-348954268.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result=chain({"query": query}, return_only_outputs=True)


"Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.  Jensen Huang  “Nothing makes me prouder than the incredible people who have  made NVIDIA the company it is today. We want our company to be  where they can do their life’s work.  “Together, we continue to drive advances in AI, HPC, gaming, creative  design, autonomous vehicles, and robotics—some of the world’s  most impactful areas.\n“I want to thank NVIDIA developers, partners, customers, and  families for the amazing work you do. Exciting new frontiers lie  ahead. Let’s seek them out together.”  We’re One Team  Tackling Challenges No One Else Can Solve NVIDIA employees are dedicated to building  technology that moves humanity forward and  to supporting the communities in which they  work and live.  We’ve been recognized as a top company in  social responsibility, and our employees are  passionate donors to hundreds of\nch